# Ensembling SGTs — Random SG Forests

`RandomSGForestClassifier` and `RandomSGForestRegressor` combine SGTs trained on bootstrap resamples. Each split considers a subset of features when feature subsampling is enabled. This notebook compares a single tree and a forest, then shows how forest settings affect test accuracy.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sgtlearn import SGTClassifier, RandomSGForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

tree = SGTClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
forest = RandomSGForestClassifier(
    n_estimators=50, max_depth=3, max_features="sqrt", random_state=0, n_jobs=-1
).fit(X_tr, y_tr)

print(f"single tree  test={tree.score(X_te, y_te):.3f}")
print(f"forest (50)  test={forest.score(X_te, y_te):.3f}")

single tree  test=0.936
forest (50)  test=0.936


## What to notice

The single tree and the 50-tree forest both reach 93.6% accuracy on this split. Their relative performance depends on the data and forest settings.

## The knobs

- `n_estimators` — number of trees. Try several counts and compare validation accuracy.
- `bootstrap` — resample rows per tree (True) for diversity; `max_samples` sets the resample size.
- `max_features` — features considered per split (`"sqrt"`, `"log2"`, int, float). Lower = more diverse trees.
- `n_jobs` — parallel tree fitting (`-1` = all cores).
- `pairwise_candidates` / `pairwise_penalty` — opt into [S²GT bivariate branching](bivariate-branching.ipynb); pairs are screened within each node's `max_features` subset.

In [2]:
# More trees, then a smaller per-split feature set for extra diversity.
for n in (5, 25, 100):
    f = RandomSGForestClassifier(
        n_estimators=n, max_depth=3, max_features="sqrt", random_state=0, n_jobs=-1
    ).fit(X_tr, y_tr)
    print(f"n_estimators={n:3d}: test={f.score(X_te, y_te):.3f}")

# bootstrap + max_samples control resampling.
f = RandomSGForestClassifier(
    n_estimators=50, max_depth=3, bootstrap=True, max_samples=0.6,
    max_features="log2", random_state=0, n_jobs=-1
).fit(X_tr, y_tr)
print(f"bootstrap 60% rows, log2 feats: test={f.score(X_te, y_te):.3f}")

n_estimators=  5: test=0.930
n_estimators= 25: test=0.953
n_estimators=100: test=0.947
bootstrap 60% rows, log2 feats: test=0.959


## How does it compare to a classic random forest?

scikit-learn's `RandomForestClassifier` bags axis-aligned CART trees (each node is a single `x < t` threshold); `RandomSGForestClassifier` bags SGTs whose nodes apply a shape function to a feature. Two things set an SGT forest apart — the shape-function splits and optional TAO refinement — so we separate them, using [phoneme](https://www.openml.org/d/1489) (nasal-vs-oral vowel classification from five continuous acoustic features that relate non-monotonically to the class).

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier

# Phoneme: 5404 samples, 5 continuous acoustic features, binary target.
Xp, yp = fetch_openml("phoneme", version=1, return_X_y=True, as_frame=False, parser="liac-arff")
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(Xp, yp, test_size=0.3, random_state=0, stratify=yp)

# Use the same split, depth, tree count, and feature sampling for all three.
# The SG forests differ only in whether they run TAO.
common = dict(n_estimators=50, max_depth=3, max_features=None, random_state=0, n_jobs=-1)
sg_arch = RandomSGForestClassifier(tao_n_runs=0, **common).fit(Xp_tr, yp_tr)
sg_tao = RandomSGForestClassifier(tao_n_runs=10, **common).fit(Xp_tr, yp_tr)
rf = RandomForestClassifier(**common).fit(Xp_tr, yp_tr)

print(f"SGForest, no TAO     test={sg_arch.score(Xp_te, yp_te):.3f}")
print(f"SGForest, with TAO   test={sg_tao.score(Xp_te, yp_te):.3f}")
print(f"sklearn RandomForest test={rf.score(Xp_te, yp_te):.3f}")

SGForest, no TAO     test=0.809
SGForest, with TAO   test=0.826
sklearn RandomForest test=0.790


On this held-out split, the SG forest scores 80.9% without TAO and 82.6% with TAO, a gain of about 1.7 percentage points. The classic random forest scores 79.0%. These results describe this split and configuration.

For a single interpretable model instead, see the [quickstart](../quickstart.rst) and [structure vs. accuracy](structure-and-accuracy.ipynb). The same pair-search controls are available on forests; see [S²GT bivariate branching](bivariate-branching.ipynb). TAO also refines every tree in a forest — see [TAO › Forests](tao.ipynb).